# 03 - Train / Test Split

**Stage:** Load (the *L* in ETL) — load the transformed table and
split it into reproducible train/test partitions for modelling.

Responsibilities:
- Load the tidy table from `data/interim/`.
- Split into train/test with a fixed `random_state` for reproducibility.
- Stratify on the target where it is a classification label.
- Persist splits to `data/processed/`.

In [ ]:
# Make the project `src` package importable from the notebooks/ dir.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

config.ensure_dirs()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet(config.TRANSFORMED_FILE)
print(f"Loaded transformed shape: {df.shape}")

## Split

In [ ]:
target = config.TARGET_COLUMN
stratify = df[target] if target in df.columns else None
if stratify is None:
    print(f"Warning: target '{target}' not found — splitting without stratification.")

train_df, test_df = train_test_split(
    df,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=stratify,
)
print(f"Train: {train_df.shape}  Test: {test_df.shape}")

## Sanity-check the split

In [ ]:
if target in df.columns:
    print("Train target distribution:")
    print(train_df[target].value_counts(normalize=True))
    print("\nTest target distribution:")
    print(test_df[target].value_counts(normalize=True))

## Persist splits

In [ ]:
train_df.to_parquet(config.TRAIN_FILE, index=False)
test_df.to_parquet(config.TEST_FILE, index=False)
print(f"Wrote {config.TRAIN_FILE}")
print(f"Wrote {config.TEST_FILE}")